# First image in JupyterLab on a GPU (optional)

> **When to use this notebook:** run it inside a GPU DevLab (JupyterLab on a GPU VM), for example on an L40S. Working from your laptop with an Endpoint? Use `first_image_endpoint.ipynb` instead.
>
> Setup (account, credits): the **Setup** section of the [module README](../README.md). Delete the DevLab when you're done: a stopped DevLab still bills its disk.

Because the model runs inside this VM's GPU memory, we can measure internal model load time (`load_s`), inspect peak VRAM (`torch.cuda.max_memory_allocated()`), and iterate on prompts, seeds, and steps without reloading weights.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "common.py").is_file())
sys.path.insert(0, str(CODE.resolve()))

# Store Hugging Face weights on the DevLab local disk:
os.environ.setdefault("HF_HOME", str(Path.home() / "hf"))

## 1. Check GPU and Install Diffusers Dependencies

Before loading weights, verify that CUDA is available and check how much free VRAM the GPU currently has. Remember that other running Jupyter kernels hold onto their GPU memory until shut down.

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.39.0", "transformers==5.13.1", "accelerate==1.14.0",
    "sentencepiece==0.2.1", "protobuf==6.33.0",
])
import torch

assert torch.cuda.is_available(), "CUDA is not available. Run this notebook on a GPU DevLab."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"], text=True))

## 2. Load the Pinned Sana Pipeline Once and Measure `load_s`

We load `Efficient-Large-Model/Sana_1600M_1024px_diffusers` pinned to a Hub commit; the prompt, seed and settings are in the same cell. Keep this kernel alive while you experiment: restarting the kernel drops the pipeline from VRAM and re-pays the load time.

In [ ]:
from generate import load_pipeline

cfg = {
    "model": "Efficient-Large-Model/Sana_1600M_1024px_diffusers",
    "model_revision": "ac0da2ff55fbe434795be0dce883042e4d49e2fc",  # pinned Hub commit
    "variant": "fp16",
    "prompt": (
        "A storybook illustration of a brass telescope glowing on snow under a starry night sky, "
        "a small decorated tree and wrapped presents, warm colors, no people, no text"
    ),
    "seed": 7,
    "params": {
        "width": 1024,
        "height": 1024,
        "num_inference_steps": 20,
        "guidance_scale": 4.5,
        "negative_prompt": "",
    },
}
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()
pipe = load_pipeline(cfg)
torch.cuda.synchronize()
load_s = time.perf_counter() - t0
print(f"Loaded {cfg['model']} in {load_s:.2f} s | VRAM after load: {torch.cuda.max_memory_allocated() / 1e9:.3f} GB")

## 3. Generate and Watch Peak VRAM and Latency

Now that the model is in GPU memory, re-run this cell with a different seed, step count or size without reloading the model. Note the generation time and peak VRAM it prints.

In [ ]:
from IPython.display import Image, display

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()
img = pipe(
    prompt=cfg["prompt"],
    **cfg["params"],
    generator=torch.Generator(device="cuda").manual_seed(cfg["seed"]),
).images[0]
torch.cuda.synchronize()
gen_s = time.perf_counter() - t0
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

preview_file = Path("runs/devlab_preview.png")
preview_file.parent.mkdir(parents=True, exist_ok=True)
img.save(preview_file)
print(f"Generation time: {gen_s:.2f} s | Peak VRAM: {peak_vram_gb:.3f} GB")
display(Image(filename=str(preview_file)))

## 4. Record a 3-Sample Run (`metrics.json`)

`generate.run()` loads the model and runs 3 sequential generations. It writes `asset-000.png`..`asset-002.png`, `metrics.json` (with `config` and `env`) and `trace.jsonl`. Cost per asset comes in lesson 05.

In [ ]:
from generate import run

del pipe
torch.cuda.empty_cache()

folder = run(cfg, n=3, mode="jupyterlab")
report = json.loads((folder / "metrics.json").read_text())
print(json.dumps({k: report[k] for k in ("status", "load_s", "latency_s", "env")}, indent=2))
print("Hashes:", [a["sha256"][:16] for a in report["assets"]])

## 5. Download Results and Stop the DevLab

Download your PNGs and `metrics.json` to your laptop, then **delete** the DevLab in the Nebius console. A stopped DevLab no longer bills compute, but it still bills its disk until you delete it.

In [ ]:
print("ACTION REQUIRED: Download your files from", folder, "and delete the DevLab in the console.")